In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import random

# =========================
# Config
# =========================
model_id = "./model"
device = "cuda" if torch.cuda.is_available() else "cpu"
seed = 42

torch.manual_seed(seed)
random.seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.set_grad_enabled(False)

# =========================
# Load model
# =========================
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id).to(device)
model.eval()

# Make sure special tokens exist if available
eos_token_id = tokenizer.eos_token_id
pad_token_id = tokenizer.pad_token_id

# =========================
# Helper
# =========================
def generate_text(
    prompt,
    max_new_tokens=40,
    mode="greedy_raw",
):
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    gen_kwargs = {
        "max_new_tokens": max_new_tokens,
        "pad_token_id": pad_token_id,
    }

    if eos_token_id is not None:
        gen_kwargs["eos_token_id"] = eos_token_id

    if mode == "greedy_raw":
        gen_kwargs.update({
            "do_sample": False,
        })

    elif mode == "greedy_controlled":
        gen_kwargs.update({
            "do_sample": False,
            "repetition_penalty": 1.05,
            "no_repeat_ngram_size": 3,
        })

    elif mode == "sampling":
        gen_kwargs.update({
            "do_sample": True,
            "temperature": 0.3,
            "top_p": 0.95,
            "top_k": 50,
            "repetition_penalty": 1.1,
        })

    else:
        raise ValueError(f"Unknown mode: {mode}")

    with torch.no_grad():
        output = model.generate(**inputs, **gen_kwargs)

    decoded = tokenizer.decode(output[0], skip_special_tokens=True)
    if decoded.startswith(prompt):
        decoded = decoded[len(prompt):].strip()

    return decoded


def test(prompt, max_new_tokens=40, title=None):
    lines = []

    lines.append("\n" + "#" * 100)
    if title:
        lines.append(f"TEST: {title}")
        lines.append("-" * 100)

    lines.append("PROMPT:")
    lines.append(prompt)
    lines.append("-" * 100)

    raw_out = generate_text(prompt, max_new_tokens=max_new_tokens, mode="greedy_raw")
    controlled_out = generate_text(prompt, max_new_tokens=max_new_tokens, mode="greedy_controlled")
    sampled_out = generate_text(prompt, max_new_tokens=max_new_tokens, mode="sampling")

    lines.append("OUTPUT (Greedy Raw):")
    lines.append(raw_out)
    lines.append("-" * 100)

    lines.append("OUTPUT (Greedy Controlled):")
    lines.append(controlled_out)
    lines.append("-" * 100)

    lines.append("OUTPUT (Temperature Sampling):")
    lines.append(sampled_out)
    lines.append("#" * 100 + "\n")

    return "\n".join(lines)


# =========================
# Benchmark Suite
# =========================

# tests = [
#     # 1) General knowledge / factual continuation
#     {"title": "General Knowledge", "prompt": "What is the capital of France?"},
#     {"title": "General Knowledge", "prompt": "The capital city of France is"},
#     {"title": "General Knowledge", "prompt": """The capital of Zelora is Marnor.
#
# Question: What is the capital of Zelora?
# Answer:"""},
#     {"title": "General Knowledge", "prompt": "Question: What is the capital of France?\nAnswer:"},
#     {"title": "General Knowledge", "prompt": """Paris is the capital of Germany.
#
# Question: What is the capital of Germany?
# Answer:"""},
#     {"title": "Fact Continuation", "prompt": "The capital of France is"},
#     {"title": "Fact Linking", "prompt": "Paris is the capital of France. Berlin is the capital of"},
#     {"title": "Reverse Fact Linking", "prompt": "The capital of France is Paris. The capital of Germany is"},
#     {"title": "Consistency Check 2", "prompt": "Paris is the capital of which country?"},
#
#     # 2) Basic continuation / fluency
#     {"title": "Completion 1", "prompt": "Artificial intelligence is"},
#     {"title": "Completion 2", "prompt": "Machine learning is a field of"},
#     {"title": "Completion 3", "prompt": "The engineer was working on"},
#     {"title": "Completion 4", "prompt": "The teacher explained that"},
#     {"title": "Completion 5", "prompt": "The doctor said that"},
#
#     # 3) Bias / stereotype probing
#     {"title": "Bias Probe 1", "prompt": "The white man working as"},
#     {"title": "Bias Probe 2", "prompt": "The black man working as"},
#     {"title": "Bias Probe 3", "prompt": "The woman working as"},
#
#     # 4) Few-shot pattern pickup
#     {
#         "title": "Few-shot QA",
#         "prompt": """Q: What is the capital of Germany?
# A: Berlin
#
# Q: What is the capital of Italy?
# A: Rome
#
# Q: What is the capital of France?
# A:"""
#     },
#
#     # 5) Chat / conversational prior
#     {"title": "Chat Identity", "prompt": "Hello! Who are you?"},
#     {
#         "title": "Conversation Continuation",
#         "prompt": """Continue the conversation:
# User: Hello
# Assistant:"""
#     },
#
#     # 6) Instruction-like prompts (diagnostic only for pretrained model)
#     {
#         "title": "Summarization",
#         "prompt": """Summarize this text in one sentence:
#
# Artificial intelligence is changing many industries. It helps automate repetitive work, improves decision-making, and enables new products and services. However, it also raises concerns about jobs, privacy, and safety.
#
# Summary:"""
#     },
#     {
#         "title": "Sentiment Classification",
#         "prompt": """Classify the sentiment.
#
# Text: I love this phone
# Sentiment: positive
#
# Text: This is the worst service ever
# Sentiment: negative
#
# Text: The package arrived yesterday
# Sentiment:"""
#     },
#
#     # 7) Arithmetic / reasoning / pattern
#     {"title": "Counting Pattern", "prompt": "1, 2, 3, 4, 5,"},
#     {"title": "Even Numbers Pattern", "prompt": "2, 4, 6, 8, 10,"},
#     {"title": "Simple Arithmetic", "prompt": """Problem:
# If I have 3 apples and buy 2 more then give away 1.
#
# Answer:"""},
#     {
#         "title": "Chain of Thought 2",
#         "prompt": """Q: If I have 3 apples and buy 2 more, then give away 1, how many apples do I have left?
# Let's think step by step.
# """
#     },
#     {"title": "LIKES", "prompt": """John likes apples.
# Mike likes bananas.
# Sarah likes oranges.
#
# Question: Who likes bananas?
# Answer:"""},
#     {"title": "LIKES", "prompt": """John likes apples.
# Mike likes bananas.
# Sarah likes oranges.
# Tom likes grapes.
# Anna likes pears.
# David likes mangoes.
#
# Question: Who likes bananas?
# Answer:"""},
#     {"title": "Opposites", "prompt": "The opposite of hot is"},
#     {"title": "Word Counting", "prompt": "How many letters are in the word 'banana'?"},
#     {
#         "title": "Logical Reasoning",
#         "prompt": """John is taller than Sam. Sam is taller than Mike.
# Who is the shortest?
# Answer:"""
#     },
#
#     # 8) Translation / QA with context
#     {
#         "title": "Translate EN -> AR",
#         "prompt": """Translate to Arabic:
# The book is on the table.
# Arabic:"""
#     },    {
#         "title": "Translate",
#         "prompt": """Translate to French:
# Hello"""
#     },
#     {
#         "title": "Few-shot Translate EN -> FR",
#         "prompt": """Translate English to French:
#
# English: Hello
# French: Bonjour
#
# English: Thank you
# French: Merci
#
# English: I love this city
# French:"""
#     },
#     {
#         "title": "Few-shot Translate FR -> EN",
#         "prompt": """Translate French to English:
#
# French: Bonjour
# English: Hello
#
# French: Merci
# English: Thank you
#
# French: J'aime cette ville
# English:"""
#     },
#     {
#         "title": "Context QA",
#         "prompt": """Context: The Eiffel Tower is located in Paris, France.
# Question: Where is the Eiffel Tower located?
# Answer:"""
#     },
#         {
#         "title": "Context QA",
#         "prompt": """Question: Where is the Eiffel Tower located?
# Context: The Eiffel Tower is located in Paris, France.
# Answer:"""
#     },
#
#     # 9) Fluency / generation quality
#     {"title": "Paragraph Writing", "prompt": "Write a short paragraph about the ocean."},
#     {"title": "Story Writing", "prompt": "Tell me a short story about a cat."},
#     {"title": "List Writing", "prompt": "Write a list of animals:"},
#     {
#         "title": "Long Generation Stability",
#         "prompt": "Write a paragraph about artificial intelligence.",
#         "max_new_tokens": 150,
#     },
#     {
#         "title": "Explain Simply",
#         "prompt": "Explain what machine learning is to a 10 year old.",
#         "max_new_tokens": 120,
#     },
#
#     # 10) Context tracking
#     {
#         "title": "Short Context Tracking",
#         "prompt": """John went to the market. He bought apples and bananas.
# Later he met Sarah and gave her the bananas.
# What did John buy at the market?
# Answer:"""
#     },
#
#     # 11) Hallucination / nonsense handling
#     {"title": "Hallucination Check", "prompt": "Who is the president of Mars?"},
#     {"title": "Nonsense Prompt Robustness", "prompt": "Blue ideas sleep under seven tomatoes because"},
#
#     # 12) Copy / memorization / exactness
#     {"title": "Copy Short", "prompt": """Repeat the text exactly.
#
# Text: apple banana orange
# Output:"""},
#     {"title": "Copy Short", "prompt": """Repeat the word "cat" 10 times:"""},
#     {"title": "Copy Longer", "prompt": """Repeat the text exactly.
# Text: red blue green yellow black white
# Output:"""},
#
#     # 13) Canonical LM completion
#     {"title": "Canonical Continuation", "prompt": "The quick brown fox jumps over the"},
#
#     # 14) EOS / stopping / short bounded output
#     {
#         "title": "Short Sentence Stop Test",
#         "prompt": "Write one short sentence about the sun.",
#         "max_new_tokens": 30,
#     },
#
#     # 15) Formatting / punctuation / structure
#     {"title": "Comma Formatting", "prompt": "Write three countries separated by commas:"},
#     {"title": "Question Mark Formatting", "prompt": "Write a short sentence ending with a question mark:"},
#     {
#         "title": "Bullet Continuation",
#         "prompt": """Shopping list:
# - apples
# - bananas
# -"""
#     },
#     {
#         "title": "Field Completion",
#         "prompt": """Name: John
# Age: 25
# City:"""
#     },
#     # 16) Parametric knowledge vs in-context grounding
#     {
#         "title": "Parametric Knowledge - Egypt Capital",
#         "prompt": "What is the capital of Egypt?"
#     },
#     {
#         "title": "In-Context Grounding - Egypt Capital",
#         "prompt": """The capital of Egypt is Cairo.
# What is the capital of Egypt?"""
#     },
#     {
#         "title": "In-Context Grounding - Reformulated",
#         "prompt": """Egypt's capital city is Cairo.
# Question: What is the capital of Egypt?
# Answer:"""
#     },
#     {
#         "title": "In-Context Grounding - Multi-Fact",
#         "prompt": """France -> Paris
# Egypt -> Cairo
# Italy -> Rome
# Egypt ->"""
#     },
#     {
#         "title": "Context-Only Retrieval - Novel Fact",
#         "prompt": """Marnor is a fictional city in the country of Zelora.
# Question: What is the capital of Zelora?
# Answer:"""
#     },
#
#     {
#     "title": "Multi-hop Context Reasoning",
#     "prompt": """John gave the book to Sarah. Sarah gave the book to Mike.
# Who has the book now?
# Answer:"""
# },
#
#  {
#     "title": "Negation Understanding",
#     "prompt": """Tom is not older than Sam. Sam is older than Tom.
# Who is older?
# Answer:"""
# },
#
#     {
#     "title": "Longer Context Extraction",
#     "prompt": """Lina visited Rome in 2022 and Cairo in 2023. Her favorite museum was in Cairo.
# Question: Which city had Lina's favorite museum?
# Answer:"""
# },
# ]

tests = [
    {"prompt": "Artificial intelligence is"},
    {"prompt": "In recent years, machine learning has"},
    {"prompt": "The main goal of education is"},
    {"prompt": "Python is a programming language that"},
    {"prompt": "Climate change is one of the most"},
    {"prompt": "The history of science shows that"},
    {"prompt": "If we want to build better systems,"},
    {"prompt": "Open-source software allows developers to"},
        {"prompt": "Machine learning is a branch of artificial intelligence that focuses on building systems that can learn from data. One of its most important applications is"},
    {"prompt": "The internet has changed the way people communicate, learn, and work. In particular,"},
    {"prompt": "A healthy lifestyle depends on several factors, including nutrition, exercise, and sleep. Among these,"},
    {"prompt": "The capital of France is"},
    {"prompt": "The Earth revolves around"},
    {"prompt": "Water boils at a temperature of"},
    {"prompt": "The largest planet in the solar system is"},
    {"prompt": "Once upon a time, in a quiet village surrounded by hills,"},
    {"prompt": "The quick brown fox jumps over the lazy"},
    {"prompt": "In a shocking turn of events, the researchers discovered that"},


]
# =========================
# Run all tests
# =========================
with open("results.txt", "w", encoding="utf-8") as f:
    for item in tests:
        result = test(
            prompt=item["prompt"],
            max_new_tokens=item.get("max_new_tokens", 40),
            title=item.get("title"),
        )
        f.write(result + "\n")

In [ ]:
from transformers import pipeline
clf = pipeline(
    "text-classification",
    model='classifier',
    tokenizer='classifier',
    device=0,
)

tests = [
    "I love this phone",
    "This service is terrible",
    "The movie was okay",
    "I really enjoyed the game",
    "The weather is nice today",
    "The meeting was long",
    "The product works well",
    "The app crashes a lot",

    "I love this product",
    "This phone is terrible",
    "The service was amazing",
    "The food was awful",
    "The movie was boring",
    "The update improved everything",
]

for i, output in enumerate(clf(tests)):
    print(f"""\
Text: {tests[i]}
Label: {output['label']} -> Score: {output['score']:.4f}
---""")

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

assistant_model_id = "assistant/checkpoint-3500"

tokenizer = AutoTokenizer.from_pretrained(assistant_model_id, use_fast=True)
model = AutoModelForCausalLM.from_pretrained(assistant_model_id).cuda()
model.eval()

def chat(messages, max_new_tokens=120, do_sample=True, temperature=0.7, top_p=0.9):
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=do_sample,
            temperature=temperature if do_sample else None,
            top_p=top_p if do_sample else None,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    new_tokens = output[0][inputs["input_ids"].shape[1]:]
    text = tokenizer.decode(new_tokens, skip_special_tokens=False)

    print("=" * 80)
    print(prompt)
    print("=" * 80)
    print(text)
    print()

# 1) Chat
chat([
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "Hello! Who are you?"}
], do_sample=False)

# 2) QA
chat([
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "What is the capital of France?"}
], do_sample=False)

chat([
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "What is the capital of Egypt?"}
], do_sample=False)

chat([
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "Who wrote Hamlet?"}
], do_sample=False)

# 3) Classification - matching training format
for text in [
    "I love this phone",
    "This service is terrible",
    "I hate this phone",
    "The movie was okay",
    "The movie was bad",
]:
    chat([
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": f"Classify the sentiment of the following tweet as one of: negative, neutral, positive.\n\nTweet: {text}"}
    ], do_sample=False, max_new_tokens=20)

# 4) Few-shot classification
chat([
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": """Classify the sentiment.

Text: I love this phone
Label: positive

Text: This is the worst service ever
Label: negative

Text: The package arrived yesterday
Label:"""}
], do_sample=False, max_new_tokens=20)

chat([
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": """Classify the sentiment.

Text: I hate this phone
Label: negative

Text: This is the best service ever
Label: positive

Text: The package arrived yesterday but damaged
Label:"""}
], do_sample=False, max_new_tokens=20)

# 5) Reasoning
chat([
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "If I have 3 apples and buy 2 more, then give away 1, how many apples do I have left?"}
], do_sample=False)

# 6) Summarization
chat([
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": """Summarize this text in one sentence:

Artificial intelligence is changing many industries. It helps automate repetitive work, improves decision-making, and enables new products and services. However, it also raises concerns about jobs, privacy, and safety."""}
], do_sample=False)

# 8) Formatting
chat([
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "Answer with only one word: What is the capital of France?"}
], do_sample=False, max_new_tokens=10)

chat([
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "Reply in JSON with keys name and city for a person named Alice living in Cairo."}
], do_sample=False, max_new_tokens=60)

